In [ ]:
library(tidyverse)
library(vroom)
library(data.table)
library(future.apply)

####### Systematic processing. ########
out_dir <- "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4/"
# Create outdir.
dir.create(out_dir, showWarnings = FALSE)


# First we look at all the samples that are available. 
input_dir1 <- "/mnt/dawnccle2/processed_data/reprocess_250221/nova230516/"
input_dir2 <- "/mnt/dawnccle2/processed_data/reprocess_250221/nova240826/"
input_dir3 <- "/mnt/dawnccle2/processed_data/reprocess_250221/nova241106/"
input_dir4 <- "/mnt/dawnccle2/processed_data/reprocess_250221/K562/"
input_dir5 <- "/mnt/dawnccle2/processed_data/reprocess_250221/nova250313/"
input_dir6 <- "/mnt/dawnccle2/processed_data/reprocess_250221/merged_250325/"

# Get all full filenames in all the input dirs. 
cellline_filenames1 <- list.files(path = input_dir1, pattern = "umi_dedup_fine_grained_idx.csv$", full.names = TRUE)
cellline_filenames2 <- list.files(path = input_dir2, pattern = "umi_dedup_fine_grained_idx.csv$", full.names = TRUE)
cellline_filenames3 <- list.files(path = input_dir3, pattern = "umi_dedup_fine_grained_idx.csv$", full.names = TRUE)
cellline_filenames4 <- list.files(path = input_dir4, pattern = "umi_dedup_fine_grained_idx.csv$", full.names = TRUE)
cellline_filenames5 <- list.files(path = input_dir5, pattern = "umi_dedup_fine_grained_idx.csv$", full.names = TRUE)
cellline_filenames6 <- list.files(path = input_dir6, pattern = "umi_dedup_fine_grained_idx.csv$", full.names = TRUE)

# Make a df of these filenames.
cellline_paths <- data.frame(filename = c(cellline_filenames1, cellline_filenames2, cellline_filenames3, cellline_filenames4, cellline_filenames5, cellline_filenames6))
# cellline_paths <- data.frame(filename = c(cellline_filenames4,cellline_filenames5))
cellline_paths <- cellline_paths %>% 
  mutate(basename = basename(filename)) %>%
  mutate(sample = str_extract(basename(filename), ".+(?=_umi_dedup)")) %>%
  mutate(condition = str_extract(sample, "^.+(?=-rep\\d)")) %>% 
  # Filter out K562_1ugNuc.
  filter(!str_detect(sample, "K562_1ugNuc")) %>%
  # filter out the samples HCC38-rep1-A04_S4, HCC38-rep2-A05_S5, HCC38-rep3-A06_S6
  # filter(!str_detect(sample, "HCC38-rep1-A04_S4|HCC38-rep2-A05_S5|HCC38-rep3-A06_S6")) %>%
  # mutate condition to strip the following: _100tfx, _150tfx, _1ugNuc, _2ugNuc.
  mutate(condition = str_replace(condition, "_\\d+tfx", "")) %>%
  mutate(condition = str_replace(condition, "_\\d+ugNuc", "")) %>% 
  # Filter out these conditions: "OSRC2", "Kelly_old", "SKNAS_Nuc"
  filter(!condition %in% c("OSRC2", "Kelly_old", "SKNAS_Nuc")) %>% 
  # Filter out filename that contain raw_47celltype/A172 or raw_47celltype/KMRC1 or 47celltype/K562_1ugNuc
  # filter(!str_detect(filename, "raw_47celltype/A172|raw_47celltype/KMRC1|47celltype/K562_1ugNuc")) %>%
  # Strip SKNAS_tfx of the _tfx.
  mutate(condition = str_replace(condition, "_tfx", "")) %>% 
  # ALso extract _Nuc.
  mutate(condition = str_replace(condition, "_Nuc", "")) %>%
  # Change the DBTR05MG to DBTRG05MG
  mutate(condition = str_replace(condition, "DBTR05MG", "DBTRG05MG")) %>%
  # Change MEWO to MeWo
  mutate(condition = str_replace(condition, "MEWO", "MeWo")) %>%
  # Change JHOM to JHOM1
  mutate(condition = str_replace(condition, "JHOM", "JHOM1")) %>%
  # Change splicelib_MEL202 to MEL202
  mutate(condition = str_replace(condition, "splicelib_MEL202", "MEL202")) %>%
  mutate(rep_old = str_extract(basename(filename), "rep\\d")) %>%
  # Create rep new, which is rep1 = rep1, rep2 = rep2, rep3 = rep3, rep4 = rep1, rep5 = rep2, rep6 = rep3.
  mutate(rep_new = case_when(
    rep_old == "rep1" ~ "rep1",
    rep_old == "rep2" ~ "rep2",
    rep_old == "rep3" ~ "rep3",
    str_detect(basename(filename), "rep4") ~ "rep1",
    str_detect(basename(filename), "rep5") ~ "rep2",
    str_detect(basename(filename), "rep6") ~ "rep3"
  )) %>% 
  mutate(sample_new = paste0(condition, "-", rep_new))

# Get the condition is NA but has K562 in the sample name.
cellline_paths_K562s <- cellline_paths %>% filter(is.na(condition) & str_detect(sample, "K562")) %>% arrange(sample)
cellline_paths_K562s$condition <- ifelse(grepl("K562_K700E", cellline_paths_K562s$sample), "K562K700E", "K562WT") 
cellline_paths_K562s$rep_old <- c("rep1", "rep2", "rep3", "rep1", "rep2", "rep3")
cellline_paths_K562s$rep_new <- c("rep1", "rep2", "rep3", "rep1", "rep2", "rep3")
cellline_paths_K562s$sample_new <- paste0(cellline_paths_K562s$condition, "-", cellline_paths_K562s$rep_new)

# Get the condition is NA and has splicelib.
cellline_paths_single_muts <- cellline_paths %>% filter(is.na(condition) & str_detect(sample, "splicelib")) %>% arrange(sample)
# Condition is everything before _splicelib
cellline_paths_single_muts$condition <- str_extract(cellline_paths_single_muts$sample, "^.*(?=_splicelib)")
# rep is splicelib_1,2. But make it rep1, rep2.
cellline_paths_single_muts$rep_old <- str_extract(cellline_paths_single_muts$sample, "splicelib_\\d")
cellline_paths_single_muts$rep_old <- str_replace(cellline_paths_single_muts$rep_old, "splicelib_", "rep")
# rep_new is rep1, rep2.
cellline_paths_single_muts$rep_new <- cellline_paths_single_muts$rep_old
cellline_paths_single_muts$sample_new <- paste0(cellline_paths_single_muts$condition, "-", cellline_paths_single_muts$rep_new)

# # merge these with the original cellline_paths.
cellline_paths_everythingelse <- cellline_paths %>% filter(!is.na(condition))
cellline_paths <- rbind(cellline_paths_K562s, cellline_paths_everythingelse, cellline_paths_single_muts)



── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘vroom’


The following objects are masked from ‘package:readr’:

    as.col_spec, col_character, col_date, col_datetime, col_double,
    col_factor, col_guess, col_integer, col_logical, col_number,
    col_skip, col_time, cols, cols_condense, cols_only, date_names,
    date_names_lang, date_names_langs, default_locale, fwf_cols,
    fwf_empty, fwf_positions, fwf_widths, locale, output_column,
    problems, spec



Attaching package: ‘data.table’


The

In [5]:
cellline_paths_filtered

filename,basename,sample,condition,rep_old,rep_new,sample_new,total_reads,total_aligned_reads,total_umi_count,duplication_rate,bc_not_found,perc_bc_not_found,skipped_reads,chimera_reads,perc_chimera_reads,unspliced_counter,minor_splice_counter,perfect_middle_exon,everything_passed_with_middle_exon
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
/mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_K700E-H04_S262_umi_dedup_fine_grained_idx.csv,K562_K700E-H04_S262_umi_dedup_fine_grained_idx.csv,K562_K700E-H04_S262,K562K700E,rep1,rep1,K562K700E-rep1,41505772,38917674,6485351,0.166642821459474,1372050,0.0330568480933206,8225067,4981475,0.180437917761697,980429,2104483,21672203,22626220
/mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_K700E-H05_S263_umi_dedup_fine_grained_idx.csv,K562_K700E-H05_S263_umi_dedup_fine_grained_idx.csv,K562_K700E-H05_S263,K562K700E,rep2,rep2,K562K700E-rep2,42735524,40157575,6221726,0.154932811555479,1424120,0.0333240327180731,8562758,5064171,0.178156637104475,1070625,2098807,22369654,23361214
/mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_K700E-H06_S264_umi_dedup_fine_grained_idx.csv,K562_K700E-H06_S264_umi_dedup_fine_grained_idx.csv,K562_K700E-H06_S264,K562K700E,rep3,rep3,K562K700E-rep3,36274891,34036200,4406776,0.129473207937431,1205193,0.0332238903212693,7064888,4282589,0.175021971799673,775212,1727235,19354731,20186276
/mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_WT-H01_S259_umi_dedup_fine_grained_idx.csv,K562_WT-H01_S259_umi_dedup_fine_grained_idx.csv,K562_WT-H01_S259,K562WT,rep1,rep1,K562WT-rep1,73478733,69208355,4722380,0.0682342471512291,2394265,0.0325844622280028,11296555,9266588,0.173900625007905,927989,3697125,42404569,44020098
/mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_WT-H02_S260_umi_dedup_fine_grained_idx.csv,K562_WT-H02_S260_umi_dedup_fine_grained_idx.csv,K562_WT-H02_S260,K562WT,rep2,rep2,K562WT-rep2,92707992,87249310,6624051,0.0759209557072715,3114571,0.0335954962760924,17474511,11029645,0.171165218579898,1185136,4151075,51390573,53408943
/mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_WT-H03_S261_umi_dedup_fine_grained_idx.csv,K562_WT-H03_S261_umi_dedup_fine_grained_idx.csv,K562_WT-H03_S261,K562WT,rep3,rep3,K562WT-rep3,53634921,50352253,4243504,0.0842763480712571,1792567,0.0334216396067778,9787663,6441872,0.171179550987275,616641,2315697,30029113,31190380
/mnt/dawnccle2/processed_data/reprocess_250221/nova230516//769P-rep1_umi_dedup_fine_grained_idx.csv,769P-rep1_umi_dedup_fine_grained_idx.csv,769P-rep1,769P,rep1,rep1,769P-rep1,43846612,41249326,2431151,0.0589379569498905,1448518,0.0330360302410594,9770035,5517950,0.193121710898869,1033852,1873043,22008462,23054446
/mnt/dawnccle2/processed_data/reprocess_250221/nova230516//769P-rep2_umi_dedup_fine_grained_idx.csv,769P-rep2_umi_dedup_fine_grained_idx.csv,769P-rep2,769P,rep2,rep2,769P-rep2,49465181,46383549,2888290,0.0622697068738746,1631880,0.0329904786965199,11211870,6419912,0.200972652099331,1137722,2089750,24364161,25524295
/mnt/dawnccle2/processed_data/reprocess_250221/nova230516//769P-rep3_umi_dedup_fine_grained_idx.csv,769P-rep3_umi_dedup_fine_grained_idx.csv,769P-rep3,769P,rep3,rep3,769P-rep3,45547717,42858467,2612690,0.0609608831785794,1524717,0.0334751574925259,10193087,5899798,0.199390816542137,1082177,1994087,22613880,23689318


In [4]:
# Extract all metrics and append to cellline_paths
metrics_list <- lapply(1:nrow(cellline_paths), function(i) {
  filename_full <- cellline_paths$filename[i]
  sample_name <- cellline_paths$sample[i]
  filename_dir <- dirname(filename_full)
  stats_log_path <- file.path(filename_dir, paste0(sample_name, "_stats_log_fine_grained_idx.txt"))
  
  if (file.exists(stats_log_path)) {
    stats_log <- read_csv(stats_log_path, col_names = FALSE)
    colnames(stats_log) <- c("metric", "count")
    
    # Convert metrics to a named list
    metric_values <- as.list(setNames(stats_log$count, stats_log$metric))
    return(metric_values)
  } else {
    return(NULL)
  }
})

# Combine metrics with cellline_paths
metrics_df <- bind_rows(metrics_list) %>%
  mutate_all(as.character)  # Ensure consistent data type

cellline_paths <- bind_cols(cellline_paths, metrics_df)

# Filter out samples that have < 1M aligned reads.
cellline_paths_filtered <- cellline_paths %>% 
  mutate(total_aligned_reads = as.integer(total_aligned_reads)) %>% 
  filter(total_aligned_reads >= 1e6) 

# Write this metadata to file.
write_csv(cellline_paths_filtered, "/mnt/dawnccle2/melange/process_fastq_250221/02_merge_and_normalize_counts/cellline_sample_metadata_v4.csv")


Rows: 13 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): X1
dbl (1): X2

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 13 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): X1
dbl (1): X2

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 13 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): X1
dbl (1): X2

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 13 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","

In [7]:
# This is the new version of normalization based on "Version 3" math.
normalize_all_non_included_reads <- function(df, chimeric_rate) {
  df_with_stats <- df %>% 
    mutate(is_included = ifelse(mode == "INCLUDED", TRUE, FALSE)) %>% 
    group_by(index) %>% 
    mutate(
      total_count = sum(count),
      total_included_count = sum(count * is_included, na.rm = TRUE) ,
      total_not_included_count = sum(count * !is_included, na.rm = TRUE)
    ) %>% 
    ungroup()
    
  df_not_included <- df_with_stats %>% 
    filter(!is_included) %>% 
    group_by(index) %>% 
    mutate(read_frac = count / sum(count)) %>% 
    ungroup() 
  
  df_not_included <- df_not_included %>%
    mutate(count_scaled = (total_not_included_count - chimeric_rate* total_included_count)/(1+ chimeric_rate) * read_frac) %>%
    # Convert all to integer.
    mutate(count_scaled = as.integer(count_scaled)) %>% 
    # If value < 0, set to 0.
    mutate(count_scaled = ifelse(count_scaled < 0, 0, count_scaled)) %>% 
    select(-is_included, -total_count, -total_included_count, -total_not_included_count, -read_frac)
  
  return(df_not_included)
}

# Merge offsets. For each index, take the biggest offset, and then assign the +/-1 offset counts to the counts of the biggest offset.
# Iterate until there are no more offsets to merge.
merge_offsets_to_ref <- function(df){
  df_sep <- df %>% separate(offset, into = c("upstream_offset", "downstream_offset", "const_offset"), sep = ":", remove = FALSE) %>% 
  mutate(upstream_offset = as.integer(upstream_offset), downstream_offset = as.integer(downstream_offset), const_offset = as.integer(const_offset))
  cat("done separating\n")
  unique_indices <- unique(df$index)
  new_df <- data.frame()
  for (idx in unique_indices){
    df_tmp_idx <- df_sep %>% filter(index == idx) %>% 
      arrange(desc(count))
    # cat("################## before merging dataframe for index:", idx, "\n")
    # print(df_tmp_idx, n = nrow(df_tmp_idx), width = Inf)
    new_df_tmp_idx <- data.frame()
    while (nrow(df_tmp_idx) > 0){
      max_offset_row <- df_tmp_idx %>% slice(1)
      max_offset_val_upstream <- max_offset_row$upstream_offset
      max_offset_val_downstream <- max_offset_row$downstream_offset
      # Get all possible offses for upstream_offset, downstream_offset. +/- 1.
      possible_offset_rows <- df_tmp_idx %>% 
      filter((upstream_offset == max_offset_val_upstream + 1 & downstream_offset == max_offset_val_downstream) |
             (upstream_offset == max_offset_val_upstream - 1 & downstream_offset == max_offset_val_downstream) |
             (upstream_offset == max_offset_val_upstream & downstream_offset == max_offset_val_downstream + 1) |
             (upstream_offset == max_offset_val_upstream & downstream_offset == max_offset_val_downstream - 1) |
             (upstream_offset == max_offset_val_upstream & downstream_offset == max_offset_val_downstream) |
             (upstream_offset == max_offset_val_upstream + 1 & downstream_offset == max_offset_val_downstream + 1) |
             (upstream_offset == max_offset_val_upstream + 1 & downstream_offset == max_offset_val_downstream - 1) |
             (upstream_offset == max_offset_val_upstream - 1 & downstream_offset == max_offset_val_downstream + 1) |
             (upstream_offset == max_offset_val_upstream - 1 & downstream_offset == max_offset_val_downstream - 1))
      # Merge the counts for these possible_offset_rows. And assign the count to the max_offset_row.
      merged_counts <- possible_offset_rows %>% pull(count) %>% sum()
      max_offset_row$count <- merged_counts
      # Remove the possible_offset_rows from df_tmp_idx. Need to be the exact pairs that are in possible_offset_rows.
      # Filter out the exact pairs that are in possible_offset_rows
      df_tmp_idx <- anti_join(df_tmp_idx, possible_offset_rows, 
                             by = c("upstream_offset", "downstream_offset", "const_offset"))
      # Add the max_offset_row to new_df_tmp_idx.
      new_df_tmp_idx <- bind_rows(new_df_tmp_idx, max_offset_row)
    }
    # cat("Final merged dataframe for index:", idx, "\n")
    #print(new_df_tmp_idx)
    new_df <- bind_rows(new_df, new_df_tmp_idx)
  }
  # Remove the upstream_offset, downstream_offset, const_offset columns.
  new_df <- new_df %>% select(-upstream_offset, -downstream_offset, -const_offset)
  return(new_df)
}

# Parallel version of merge_offsets_to_ref.
merge_offsets_to_ref_parallel <- function(df, workers = 64) {
  plan(multisession, workers = workers)

  # Preprocess and split
  df_sep <- df %>%
    separate(offset, into = c("upstream_offset", "downstream_offset", "const_offset"), sep = ":", remove = FALSE) %>%
    mutate(across(c(upstream_offset, downstream_offset, const_offset), as.integer)) %>%
    as.data.table()

  cat("done separating\n")

  # Split by index (parallelizable units)
  df_list <- split(df_sep, by = "index")
  # Greedy merge function (same as before, per index)
  greedy_merge_index <- function(df_idx) {
    setorder(df_idx, -count)
    merged <- data.table()

    while (nrow(df_idx) > 0) {
      anchor <- df_idx[1]
      u0 <- anchor$upstream_offset
      d0 <- anchor$downstream_offset
      c0 <- anchor$const_offset

      neighbors <- df_idx[
        upstream_offset %between% c(u0 - 1, u0 + 1) &
        downstream_offset %between% c(d0 - 1, d0 + 1)
      ]

      anchor$count <- sum(neighbors$count)
      merged <- rbind(merged, anchor, use.names = TRUE)
      df_idx <- fsetdiff(df_idx, neighbors)
    }

    return(merged)
  }

  # Run in parallel across indices
  results <- future_lapply(df_list, greedy_merge_index)

  # Combine and clean
  out <- rbindlist(results)
  out[, c("upstream_offset", "downstream_offset", "const_offset") := NULL]
  return(out[])
}

# Get all unique sample_new names.
unique_samples <- cellline_paths_filtered %>% group_by(sample_new) %>% summarise(n=n()) %>% ungroup() 
unique_sample_names <- unique(cellline_paths_filtered$sample_new)

# merge_offsets_to_ref_parallel(tmp_included_sequences_only)


In [8]:
for (sample_tmp in unique_sample_names[1]){
  # Get all the filepaths with that sample name.
  sample_filepaths <- cellline_paths_filtered %>% filter(sample_new == sample_tmp) %>% pull(filename)
  tmp_out <- data.frame()
  for (filepath in sample_filepaths) {
    base::print(paste("Processing", filepath))
    # Get the parent folder name from the filepath. (not full path)
    parent_folder <- basename(dirname(filepath))
    # Get basename and strip _umi_dedup_fine_grained_idx.csv
    filename_basename <- basename(filepath) %>% str_extract(".+(?=_umi_dedup_fine_grained_idx.csv)")
    
    # Read in the tsv file.
    tmp <- vroom(filepath, id = "filename", delim = ",")
    
    # Get chimeric rate from the metadata table. 
    chimeric_rate <- cellline_paths_filtered %>% 
      filter(filename == filepath) %>% 
      pull(perc_chimera_reads)
    base::print(paste("chimeric_rate:", chimeric_rate))
    
    # Separate the "index" column into id, mode, offset, insert_size based on __ separator.
    tmp_to_ref <- tmp %>% 
      separate(index, into = c("index", "mode", "offset", "insert_size"), sep = "__", remove = FALSE)
    
    # Split into included (need offset merging), and everything else (need adjustment).
    # We just directly adjust the counts for non-included as count_scaled = count * (1 - chimeric_rate).
    tmp_included_sequences_only <- tmp_to_ref %>% filter(mode == "INCLUDED") 
    tmp_included_merged_offsets <- merge_offsets_to_ref(tmp_included_sequences_only) %>% mutate(count_scaled = count)
    # Normalize the non-included reads.  
    tmp_everything_else <- normalize_all_non_included_reads(tmp_to_ref, as.numeric(chimeric_rate))
    
    tmp_final <- bind_rows(tmp_included_merged_offsets, tmp_everything_else)

    # Write tmp_final to file.
    fwrite(tmp_final, file.path(out_dir, paste0(parent_folder, "-individual-", filename_basename, "_umi_dedup_normalized_tmp.tsv")))

    tmp_out <- bind_rows(tmp_out, tmp_final)
  }
  
  # Group the tmp_out.
  tmp_grouped <- tmp_out %>% group_by(index, mode, offset) %>%
    summarise(count = sum(count), 
              count_scaled = sum(count_scaled)) %>% 
    ungroup()
  
  # Write to outdir. 
  base::print(paste("Writing to", file.path(out_dir, paste0(sample_tmp, "_umi_dedup_normalized.tsv"))))
  fwrite(tmp_grouped, file.path(out_dir, paste0(sample_tmp, "_umi_dedup_normalized.tsv")))
}

[1] "Processing /mnt/dawnccle2/processed_data/reprocess_250221/K562//K562_K700E-H04_S262_umi_dedup_fine_grained_idx.csv"


Rows: 218495 Columns: 3
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): index
dbl (1): count

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


[1] "chimeric_rate: 0.173841153685578"
done separating


`summarise()` has grouped output by 'index', 'mode'. You can override using the
`.groups` argument.


[1] "Writing to /mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4//K562K700E-rep1_umi_dedup_normalized.tsv"


In [10]:
# Merge samples into one file. 
library(tidyverse)
library(vroom)
library(data.table)

out_dir <- "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate/"
dir.create(out_dir, showWarnings = FALSE)

process_samples <- function(input_dir, sample_type, out_dir) {
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  
  input_filenames <- list.files(path = input_dir, pattern = "umi_dedup_normalized.tsv$", full.names = TRUE)
  
  # Precompute sample and condition for each file
  file_metadata <- tibble(
    filename = input_filenames,
    sample = basename(input_filenames) %>% str_extract(".+(?=_umi_dedup_normalized.tsv)"),
    condition = str_extract(basename(input_filenames), "^.+(?=-rep\\d)")
  )
  
  # Read data and attach metadata
  all_files_df <- map_dfr(seq_along(file_metadata$filename), function(i) {
    df <- vroom(file_metadata$filename[i], delim = ",")
    df$sample <- file_metadata$sample[i]
    df$condition <- file_metadata$condition[i]
    df
  })
  
  fwrite(all_files_df, file.path(out_dir, paste0(sample_type, "_all_samples_raw_counts.csv")))
}

# Define input and output directories
input_output_mapping <- list(
  list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate/MUT2", 
       sample_type = "MUT2", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate/")
)

# Process each sample type
walk(input_output_mapping, ~process_samples(.x$input_dir, .x$sample_type, .x$out_dir))

####################################
# Also process for the other folder.
####################################

out_dir <- "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate_considering_included/"
dir.create(out_dir, showWarnings = FALSE)

process_samples <- function(input_dir, sample_type, out_dir) {
  dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)
  
  input_filenames <- list.files(path = input_dir, pattern = "umi_dedup_normalized.tsv$", full.names = TRUE)
  
  # Precompute sample and condition for each file
  file_metadata <- tibble(
    filename = input_filenames,
    sample = basename(input_filenames) %>% str_extract(".+(?=_umi_dedup_normalized.tsv)"),
    condition = str_extract(basename(input_filenames), "^.+(?=-rep\\d)")
  )
  
  # Read data and attach metadata
  all_files_df <- map_dfr(seq_along(file_metadata$filename), function(i) {
    df <- vroom(file_metadata$filename[i], delim = ",")
    df$sample <- file_metadata$sample[i]
    df$condition <- file_metadata$condition[i]
    df
  })
  
  fwrite(all_files_df, file.path(out_dir, paste0(sample_type, "_all_samples_raw_counts.csv")))
}

# Define input and output directories
input_output_mapping <- list(
  list(input_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate_considering_included/MUT2", 
       sample_type = "MUT2", 
       out_dir = "/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate_considering_included/")
)

# Process each sample type
walk(input_output_mapping, ~process_samples(.x$input_dir, .x$sample_type, .x$out_dir))

Warning message in fwrite(all_files_df, file.path(out_dir, paste0(sample_type, "_all_samples_raw_counts.csv"))):
“Input has no columns; doing nothing.
If you intended to overwrite the file at /mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_chimeric_rate//MUT2_all_samples_raw_counts.csv with an empty one, please use file.remove first.”
Rows: 248768 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): index, mode, offset
dbl (2): count, count_scaled

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 260455 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): index, mode, offset
dbl (2): count, count_scaled

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to q